In [ ]:
# Config
OPENAI_API_KEY = "..."

In [ ]:
from abc import ABC, abstractclassmethod

class LLMClient(ABC):
    @abstractclassmethod
    def chat_complete(self, message, model: str) -> str:
        pass

In [7]:
from openai import OpenAI


class ChatGPTClient(LLMClient):
    def __init__(self):
        self.client = OpenAI(
            api_key=OPENAI_API_KEY
        )
    
    def chat_complete(self, messages, model: str="gpt-4o-mini") -> str:
        response = self.client.chat.completions.create(
            model=model,
            messages=messages
        )

        return response.choices[0].message.content

In [20]:
from typing import Tuple

url = "https://x.com/gdb/status/2019566641491963946"

def summarize_en_prompt(post_content: str) -> Tuple[str, str]:
    system = (
        "You are a senior software engineer and technical writer. "
        "You produce accurate, high-signal technical summaries for professional communities."
    )

    user_prompt = f"""
    You are summarizing a technical post for a professional engineering audience
    (e.g. LinkedIn, X, technical community blogs).

    Your tasks:

    1) Generate a clear, informative TITLE for the post.
    - Concise
    - Technical
    - No hype or clickbait

    2) Write a technical summary that:
    - Highlights the core technical insight or problem
    - Extracts key technical points
    - States assumptions or constraints (if any)
    - Mentions trade-offs or limitations (if discussed)
    - Ends with a practical takeaway
    - Put {url} for reference at the end of the summerized post.

    Tone & style:
    - Professional, technical, and confident
    - Engaging enough to make engineers want to read the original post
    - No marketing fluff, no exaggeration

    Rules:
    - Preserve original technical terminology
    - Do NOT simplify technical concepts
    - Do NOT add new information or speculation
    - If something is not stated, say "Not specified"

    Output format:

    Title:
    <one line>

    Summary:
    <one cohesive paragraph, technical and engaging>

    Post:
    <<<
    {post_content}
    >>>
    """

    return system, user_prompt



def translate_vi_prompt(text: str) -> Tuple[str, str]:
    system = (
        "You are a professional technical translator specializing in "
        "Vietnamese software engineering content."
    )

    user_prompt = f"""
        Translate the following text into Vietnamese.

        Rules:
            - Use professional technical Vietnamese
            - Keep commonly used English technical terms in parentheses
            - Do NOT add explanations or opinions
            - Do NOT simplify concepts
            - Keep structure unchanged
            - Maintain technical tone
        Text:
        <<<
        {text}
        >>>
        """
    return system, user_prompt


def rewrite_vn_prompt(text: str) -> Tuple[str, str]:
    system = (
        "You adapt technical content for Vietnamese engineering communities "
        "while preserving accuracy and rigor."
    )

    user_prompt = f"""
        Rewrite the following Vietnamese technical text.

        Goals:
        - Natural Vietnamese for engineers
        - Technically precise
        - Professional but engaging
        - Encourage readers to explore the full post

        Do NOT:
        - Change technical meaning
        - Add new claims

        Text:
        <<<
        {text}
        >>>
        """
    return system, user_prompt

In [21]:
class BaseExecutor:
    model: str

    def run(self, system_prompt, user_prompt, temperature=0.2) -> str:
        raise NotImplementedError


class LLMExecutor(BaseExecutor):
    def __init__(self, client, model: str):
        self.client = client
        self.model = model

    def run(self, system_prompt, user_prompt, temperature=0.2):
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
        return self.client.chat_complete(messages, self.model)

In [24]:
import time
from contextlib import contextmanager

@contextmanager
def measure_latency(label: str, metrics: dict, model: str):
    start = time.perf_counter()
    yield
    metrics[label] = {
        "perf": round(time.perf_counter() - start, 3),
        "model": model
    }
from typing import List, Tuple, Callable

def summarize_english(executor: BaseExecutor, post: str) -> str:
    system, user = summarize_en_prompt(post)
    return executor.run(
        system_prompt=system,
        user_prompt=user,
        temperature=0.2,
    )


def translate_vietnamese(executor: BaseExecutor, text: str) -> str:
    system, user = translate_vi_prompt(text)
    return executor.run(
        system_prompt=system,
        user_prompt=user,
        temperature=0.2,
    )


def rewrite_vn_community(executor: BaseExecutor, text: str) -> str:
    system, user = rewrite_vn_prompt(text)
    return executor.run(
        system_prompt=system,
        user_prompt=user,
        temperature=0.3,
    )


def summerize_posts(post_content: str, steps: List[Tuple[str, Callable]]):
    current_text  = post_content
    metrics = {}
    for label, step_fnc, executor in steps:
        with measure_latency(label, metrics, executor.model):
            current_text = step_fnc(executor, current_text)
    metrics["total"] = round(
        sum(step["perf"] for step in metrics.values()),
        3
    )    
    return {
        "result": current_text, 
        "metrics": metrics
    }

In [38]:
MODEL_COMBINATIONS = [
    ("gpt-4o-mini", "gpt-4o-mini", "gpt-4o-mini"),
    # cheaper / fast
    ("gpt-4.1-mini", "gpt-4.1-mini", "gpt-4.1-mini"),

    # mixed
    ("gpt-4.1", "gpt-4.1-mini", "gpt-4.1"),

    # strongest
    ("gpt-4.1", "gpt-4.1", "gpt-4.1"),
    
    ("gpt-4.1-mini", "gpt-4.1-mini", "gpt-4.1"),
]

def build_executors(client, models):
    summarize_model, translate_model, rewrite_model = models

    return {
        "summarize": LLMExecutor(client, summarize_model),
        "translate": LLMExecutor(client, translate_model),
        "rewrite": LLMExecutor(client, rewrite_model),
    }

def build_steps(executors):
    return [
        ("summarize_en", summarize_english, executors["summarize"]),
        ("translate_vn", translate_vietnamese, executors["translate"]),
        ("rewrite_technical_vn", rewrite_vn_community, executors["rewrite"]),
    ]

def run_all_combinations(post: str):
    gpt_client = ChatGPTClient()
    results = []

    for models in MODEL_COMBINATIONS:
        executors = build_executors(gpt_client, models)
        steps = build_steps(executors)

        metrics = {}
        text = post

        for label, fn, executor in steps:
            with measure_latency(label, metrics, executor.model):
                text = fn(executor, text)

        metrics["total"] = round(
            sum(step["perf"] for step in metrics.values() if isinstance(step, dict)),
            3
        )

        results.append({
            "models": models,
            "output": text,
            "metrics": metrics,
        })

    return results

In [32]:
gpt_client = ChatGPTClient()

summarize_executor = LLMExecutor(
    client=gpt_client,
    model="gpt-4.1"        # strongest
)

translate_executor = LLMExecutor(
    client=gpt_client,
    model="gpt-4.1-mini"   # cheaper OK
)

rewrite_executor = LLMExecutor(
    client=gpt_client,
    model="gpt-4.1"        # fluency matters
)

steps = [
    ("summerize_en", summarize_english, summarize_executor),
    ("translate_vn", translate_vietnamese, translate_executor),
    ("rewrite_technical_vn", rewrite_vn_community, rewrite_executor),
]

In [39]:
post_content = (
    "We estimate that GPT-5.2 with `high` (not `xhigh`) reasoning effort has a 50%-time-horizon of around 6.6 hrs (95% CI of 3 hr 20 min to 17 hr 30 min) on our expanded suite of software tasks."
    "This is the highest estimate for a time horizon measurement we have reported to date."
)
run_all_combinations(post_content)


[{'models': ('gpt-4o-mini', 'gpt-4o-mini', 'gpt-4o-mini'),
  'output': '**Tiêu đề:**  \nƯớc lượng Chân trời Thời gian cho GPT-5.2 với Nỗ lực Lập luận Cao\n\n**Tóm tắt:**  \nBài viết này cung cấp một ước lượng về chân trời thời gian cho GPT-5.2 khi hoạt động với nỗ lực lập luận ở mức `cao`. Kết quả cho thấy chân trời thời gian trung vị hiệu quả khoảng 6,6 giờ, với khoảng tin cậy 95% dao động từ 3 giờ 20 phút đến 17 giờ 30 phút. Chỉ số này đại diện cho ước lượng chân trời thời gian cao nhất được báo cáo trong bối cảnh các nhiệm vụ phần mềm được đánh giá. Những phát hiện này chỉ ra khả năng cải thiện của mô hình trong các nhiệm vụ lập luận so với các phiên bản trước đó. Các giả định về tính nhất quán của nỗ lực lập luận cũng như các đặc điểm của những nhiệm vụ phần mềm trong quá trình đánh giá được thảo luận nhưng chưa được cụ thể hóa chi tiết. Ngoài ra, các giới hạn hoặc thỏa hiệp liên quan đến cài đặt nỗ lực lập luận này, chẳng hạn như tác động tiềm tàng đến độ chính xác của phản hồi ho

In [40]:
post_content = """Claude Opus 4.6 & GPT Codex 5.3 out today, and OpenClaw recently. And I'm sure xAI/Grok & Google/Gemini will soon be out with more. What an exciting time to build stuff!

I'm walking around with a smile, happy & sleep-deprived 🤣

Next week, I'll come to SF/Bay Area for a few days/weeks/months to hang out & build some stuff ;-)

Looking to focus on programming, and contribute to good engineering teams. But occasionally socialize, in as much as my introvert brain allows.

2026 is going to be fun (and wild), LFG!

PS: I'm doing a deep-dive podcast on OpenClaw with its creator (
@steipete
) soon. Let me know if you have questions."""

run_all_combinations(post_content)

[{'models': ('gpt-4o-mini', 'gpt-4o-mini', 'gpt-4o-mini'),
  'output': 'Tiêu đề:\nCông nghệ AI mới nổi: Claude Opus 4.6, GPT Codex 5.3, và OpenClaw\n\nTóm tắt:\nBài viết này thông báo về sự ra mắt của một số mô hình AI nổi bật, bao gồm Claude Opus 4.6, GPT Codex 5.3 và OpenClaw. Nó cũng đề cập đến các bản cập nhật dự kiến từ xAI/Grok và Google/Gemini. Tác giả bày tỏ sự phấn khích trước những bước tiến hiện tại trong lĩnh vực AI và nhấn mạnh tiềm năng của các mô hình này trong việc đổi mới phát triển phần mềm. Có sự cam kết rõ ràng về việc tham gia sâu vào lập trình trong các nhóm kỹ thuật chất lượng cũng như xây dựng cộng đồng. Tác giả dự định sẽ thảo luận về OpenClaw trong một podcast sắp tới và khuyến khích các câu hỏi từ các đồng nghiệp quan tâm. Mặc dù không cung cấp nhiều chi tiết kỹ thuật trong bài viết này, nhưng tác giả tin rằng cộng đồng sẽ quan tâm đến những cải tiến trong công nghệ AI. Đáng chú ý, không có hạn chế nào được đề cập liên quan đến các công nghệ này. Điều quan tr

In [41]:
post_content = """Software development is undergoing a renaissance in front of our eyes.

If you haven't used the tools recently, you likely are underestimating what you're missing. Since December, there's been a step function improvement in what tools like Codex can do. Some great engineers at OpenAI yesterday told me that their job has fundamentally changed since December. Prior to then, they could use Codex for unit tests; now it writes essentially all the code and does a great deal of their operations and debugging. Not everyone has yet made that leap, but it's usually because of factors besides the capability of the model.

Every company faces the same opportunity now, and navigating it well — just like with cloud computing or the Internet — requires careful thought. This post shares how OpenAI is currently approaching retooling our teams towards agentic software development. We're still learning and iterating, but here's how we're thinking about it right now:

As a first step, by March 31st, we're aiming that:

(1) For any technical task, the tool of first resort for humans is interacting with an agent rather than using an editor or terminal.
(2) The default way humans utilize agents is explicitly evaluated as safe, but also productive enough that most workflows do not need additional permissions.

In order to get there, here's what we recommended to the team a few weeks ago:

1. Take the time to try out the tools. The tools do sell themselves — many people have had amazing experiences with 5.2 in Codex, after having churned from codex web a few months ago. But many people are also so busy they haven't had a chance to try Codex yet or got stuck thinking \"is there any way it could do X\" rather than just trying.
  - Designate an \"agents captain\" for your team — the primary person responsible for thinking about how agents can be brought into the teams' workflow.
  - Share experiences or questions in a few designated internal channels
  - Take a day for a company-wide Codex hackathon

2. Create skills and AGENTS[.md].
  - Create and maintain an AGENTS[.md] for any project you work on; update the AGENTS[.md] whenever the agent does something wrong or struggles with a task.
  - Write skills for anything that you get Codex to do, and commit it to the skills directory in a shared repository

3. Inventory and make accessible any internal tools.
  - Maintain a list of tools that your team relies on, and make sure someone takes point on making it agent-accessible (such as via a CLI or MCP server).

4. Structure codebases to be agent-first. With the models changing so fast, this is still somewhat untrodden ground, and will require some exploration.
  - Write tests which are quick to run, and create high-quality interfaces between components.

5. Say no to slop. Managing AI generated code at scale is an emerging problem, and will require new processes and conventions to keep code quality high
  - Ensure that some human is accountable for any code that gets merged. As a code reviewer, maintain at least the same bar as you would for human-written code, and make sure the author understands what they're submitting.

6. Work on basic infra. There's a lot of room for everyone to build basic infrastructure, which can be guided by internal user feedback. The core tools are getting a lot better and more usable, but there's a lot of infrastructure that currently go around the tools, such as observability, tracking not just the committed code but the agent trajectories that led to them, and central management of the tools that agents are able to use.

Overall, adopting tools like Codex is not just a technical but also a deep cultural change, with a lot of downstream implications to figure out. We encourage every manager to drive this with their team, and to think through other action items — for example, per item 5 above, what else can prevent a lot of \"functionally-correct but poorly-maintainable code\" from creeping into codebases.
"""

run_all_combinations(post_content)

[{'models': ('gpt-4o-mini', 'gpt-4o-mini', 'gpt-4o-mini'),
  'output': '---\n**Tiêu đề:**  \nTích hợp Các Đại lý AI vào Quy trình Phát triển Phần mềm\n\n**Tóm tắt:**  \nBài viết này sẽ đề cập đến xu hướng ngày càng phát triển của OpenAI trong việc tích hợp các đại lý AI vào quy trình phát triển phần mềm. Đặc biệt, chúng ta sẽ xem xét những bước tiến đáng kể mà Codex đã đạt được từ tháng 12. Mục tiêu là thiết lập các tương tác theo hướng đại lý cho các nhiệm vụ kỹ thuật trước ngày 31 tháng 3, với một tư duy nhấn mạnh vào an toàn và năng suất mà không yêu cầu quá nhiều sự cấp phép. Những khuyến nghị quan trọng dành cho các nhóm bao gồm thử nghiệm với các công cụ mới, chỉ định một đại lý trưởng, duy trì tài liệu AGENTS[.md], cấu trúc mã nguồn để tạo điều kiện cho các tương tác của đại lý và ưu tiên chất lượng mã để giảm thiểu rủi ro liên quan đến mã do AI tạo ra. Sự chuyển đổi sang quy trình phát triển phần mềm hỗ trợ bởi đại lý được xem như một thay đổi văn hóa sâu sắc, đòi hỏi quản lý p